# Rows of Tables
Now that we have a qualitative understanding of nearest neighbor classification, it's time to implement our classifier.

Until this chapter, we have worked mostly with single columns of tables. But now we have to see whether one *individual* is "close" to another. Data for individuals are contained in *rows* of tables.

So let's start by taking a closer look at rows.

In [1]:
import polars as pl
import numpy as np
from datascience import make_array
import hvplot.polars
import holoviews as hv
path_data = '../../../assets/data/'

# POLARS_CHANGE: hvplot colors each point by its Color column; this maps the Data 8 color names to Data 8 blue and gold.
color_map = {'darkblue': '#002fa7', 'gold': '#706700'}

In [2]:
def standard_units(x):
    """POLARS_CHANGE: x is a column (a Polars Series), so this uses Series methods. std(ddof=0) matches np.std."""
    return (x - x.mean())/x.std(ddof=0)

Here is the original table `ckd` containing data on patients who were tested for chronic kidney disease.

In [3]:
ckd = pl.read_csv(path_data + 'ckd.csv').rename({'Blood Glucose Random': 'Glucose'})
# POLARS_CHANGE: two White Blood Cell Count entries start with a tab, so Polars reads that column as text.
# Strip the tab and convert the column to integers.
ckd = ckd.with_columns(pl.col('White Blood Cell Count').str.strip_chars().cast(pl.Int64))

The data corresponding to the first patient is in row 0 of the table, consistent with Python's indexing system. The DataFrame method `row` accesses the row by taking the index of the row as its argument:

In [4]:
# POLARS_CHANGE: named=True returns the row as a dictionary, so the display shows each label with its value.
ckd.row(0, named=True)

{'Age': 48,
 'Blood Pressure': 70,
 'Specific Gravity': 1.005,
 'Albumin': 4,
 'Sugar': 0,
 'Red Blood Cells': 'normal',
 'Pus Cell': 'abnormal',
 'Pus Cell clumps': 'present',
 'Bacteria': 'notpresent',
 'Glucose': 117,
 'Blood Urea': 56,
 'Serum Creatinine': 3.8,
 'Sodium': 111,
 'Potassium': 2.5,
 'Hemoglobin': 11.2,
 'Packed Cell Volume': 32,
 'White Blood Cell Count': 6700,
 'Red Blood Cell Count': 3.9,
 'Hypertension': 'yes',
 'Diabetes Mellitus': 'no',
 'Coronary Artery Disease': 'no',
 'Appetite': 'poor',
 'Pedal Edema': 'yes',
 'Anemia': 'yes',
 'Class': 1}

Polars returns a row as a *tuple* of values. With `named=True`, as above, it returns a dictionary instead, so the display shows not only the values in the row but also the labels of the corresponding columns.

Rows are in general **not arrays**, as their elements can be of different types. For example, some of the elements of the row above are strings (like `'abnormal'`) and some are numerical. So the row can't be converted into an array.

However, rows share some characteristics with arrays.  You can use square brackets to access a particular element of a row by its position. For example, to access the Albumin level of Patient 0, we can look at the labels in the printout of the row above to find that it's item 3:

**POLARS_CHANGE:** Data 8 rows are *row objects* that show their labels and use `item`. A Polars row is a tuple (`row[3]`), or a dictionary with `named=True`.

In [5]:
# POLARS_CHANGE: ckd.row(0) is a tuple; square brackets take the item at position 3.
ckd.row(0)[3]

4

## Converting Rows to Arrays (When Possible)
Rows whose elements are all numerical (or all strings) can be converted to arrays.  Converting a row to an array gives us access to arithmetic operations and other nice NumPy functions, so it is often useful.

Recall that in the previous section we tried to classify the patients as 'CKD' or 'not CKD', based on two attributes `Hemoglobin` and `Glucose`, both measured in standard units. 

In [6]:
ckd = pl.DataFrame([
    standard_units(ckd.get_column('Hemoglobin')),
    standard_units(ckd.get_column('Glucose')),
    ckd.get_column('Class'),
])

color_table = pl.DataFrame([
    pl.Series('Class', make_array(1, 0)),
    pl.Series('Color', make_array('darkblue', 'gold')),
])
# POLARS_CHANGE: Data 8 join sorts the rows by the joined column. Polars keeps the order of the left table, so sort by Class.
ckd = ckd.join(color_table, on='Class', maintain_order='left').sort('Class', maintain_order=True)
ckd

Hemoglobin,Glucose,Class,Color
f64,f64,i64,str
0.456884,0.133751,0,"""gold"""
1.153004,-0.947597,0,"""gold"""
0.770138,-0.762223,0,"""gold"""
0.596108,-0.190654,0,"""gold"""
-0.239236,-0.49961,0,"""gold"""
…,…,…,…
-1.42264,1.693982,1,"""darkblue"""
-1.492252,1.27689,1,"""darkblue"""
-1.7707,-0.391475,1,"""darkblue"""


Here is a scatter plot of the two attributes, along with a red point corresponding to Alice, a new patient. Her value of hemoglobin is 0 (that is, at the average) and glucose 1.1 (that is, 1.1 SDs above average).

In [7]:
alice = make_array(0, 1.1)
(
    ckd.hvplot.scatter(x='Hemoglobin', y='Glucose', color='Color', cmap=color_map)
    * hv.Scatter(([alice.item(0)], [alice.item(1)])).opts(color='red', size=8)
)

Scatterplot with 'Hemoglobin' on the x-axis and 'Glucose' on the y-axis. Data points are shown in dark blue or in gold. The dark blue data points exist all over the graph, but not where the gold data points do, from about x=0 to x=1.5 and y values between -1 and just above 0. A red data point exists at about (0, 1).

To find the distance between Alice's point and any of the other points, we only need the values of the attributes:

In [8]:
ckd_attributes = ckd.select('Hemoglobin', 'Glucose')

In [9]:
ckd_attributes

Hemoglobin,Glucose
f64,f64
0.456884,0.133751
1.153004,-0.947597
0.770138,-0.762223
0.596108,-0.190654
-0.239236,-0.49961
…,…
-1.42264,1.693982
-1.492252,1.27689
-1.7707,-0.391475


Each row consists of the coordinates of one point in our training sample. **Because the rows now consist only of numerical values**, it is possible to convert them to arrays.  For this, we use the function `np.array`, which converts any kind of sequential object, like a row, to an array. (Our old friend `make_array` is for *creating* arrays, not for *converting* other kinds of sequences to arrays.)

In [10]:
ckd_attributes.row(3)

(0.5961076648232668, -0.19065363034327712)

In [11]:
np.array(ckd_attributes.row(3))

array([ 0.59610766, -0.19065363])

This is very handy because we can now use array operations on the data in each row.

## Distance Between Points When There are Two Attributes
The main calculation we need to do is to find the distance between Alice's point and any other point. For this, the first thing we need is a way to compute the distance between any pair of points. 

How do we do this?  In 2-dimensional space, it's pretty easy.  If we have a point at coordinates $(x_0,y_0)$ and another at $(x_1,y_1)$, the distance between them is

$$
D = \sqrt{(x_0-x_1)^2 + (y_0-y_1)^2}
$$

(Where did this come from?  It comes from the Pythogorean theorem: we have a right triangle with side lengths $x_0-x_1$ and $y_0-y_1$, and we want to find the length of the hypotenuse.)



In the next section we'll see that this formula has a straightforward extension when there are more than two attributes. For now, let's use the formula and array operations to find the distance between Alice and the patient in Row 3.

In [12]:
patient3 = np.array(ckd_attributes.row(3))
alice, patient3

(array([ 0. ,  1.1]), array([ 0.59610766, -0.19065363]))

In [13]:
distance = np.sqrt(np.sum((alice - patient3)**2))
distance

1.4216649188818471

We're going to need the distance between Alice and a bunch of points, so let's write a function called `distance` that computes the distance between any pair of points. The function will take two arrays, each containing the $(x, y)$ coordinates of a point.  (Remember, those are really the Hemoglobin and Glucose levels of a patient.)

In [14]:
def distance(point1, point2):
    """Returns the Euclidean distance between point1 and point2.
    
    Each argument is an array containing the coordinates of a point."""
    return np.sqrt(np.sum((point1 - point2)**2))

In [15]:
distance(alice, patient3)

1.4216649188818471

We have begun to build our classifier: the `distance` function is the first building block. Now let's work on the next piece.

## Applying a Function to an Entire Row
Recall that if you want to apply a function to each element of a column of a table, one way to do that is by the call `table_name.get_column(column_label).map_elements(function_name, return_dtype=float)`. This evaluates to a Series consisting of the values of the function when we call it on each element of the column. So each entry of the Series is based on the corresponding row of the table.

To pass the entire row to the function instead, we loop over the rows with `table_name.iter_rows()`, which gives each row as a tuple, and collect the results in an array. Let's see how this works on a very small table `t` containing the information about the first five patients in the training sample.

**POLARS_CHANGE:** Data 8 `apply` without a column label passes each row to the function. Polars has no such `apply`; `iter_rows` in a `for` loop does the same job.

In [16]:
t = ckd_attributes.head(5)
t

Hemoglobin,Glucose
f64,f64
0.456884,0.133751
1.153004,-0.947597
0.770138,-0.762223
0.596108,-0.190654
-0.239236,-0.49961


Just as an example, suppose that for each patient we want to know how unusual their most unusual attribute is.  Concretely, if a patient's hemoglobin level is further from the average than her glucose level, we want to know how far it is from the average.  If her glucose level is further from the average than her hemoglobin level, we want to know how far that is from the average instead.

That's the same as taking the maximum of the absolute values of the two quantities. To do this for a particular row, we can convert the row to an array and use array operations.

In [17]:
def max_abs(row):
    return np.max(np.abs(np.array(row)))

In [18]:
max_abs(t.row(4))

0.49961028259186974

And now we can apply `max_abs` to each row of the table `t`:

In [19]:
# POLARS_CHANGE: iter_rows passes each row of t to max_abs as a tuple; collect the results in an array.
max_abs_values = make_array()
for row in t.iter_rows():
    max_abs_values = np.append(max_abs_values, max_abs(row))
max_abs_values

array([ 0.4568837 ,  1.15300352,  0.77013762,  0.59610766,  0.49961028])

This way of applying a function to every row will help us create the next building block of our classifier.

## Alice's $k$ Nearest Neighbors
If we want to classify Alice using a k-nearest neighbor classifier, we have to identify her $k$ nearest neighbors. What are the steps in this process? Suppose $k = 5$. Then the steps are:
- **Step 1.** Find the distance between Alice and each point in the training sample.
- **Step 2.** Sort the data table in increasing order of the distances.
- **Step 3.** Take the top 5 rows of the sorted table.

Steps 2 and 3 seem straightforward, provided we have the distances. So let's focus on Step 1.

Here's Alice:

In [20]:
alice

array([ 0. ,  1.1])

What we need is a function that finds the distance between Alice and another point whose coordinates are contained in a row. The function `distance` returns the distance between any two points whose coordinates are in arrays. We can use that to define `distance_from_alice`, which takes a row as its argument and returns the distance between that row and Alice.

In [21]:
def distance_from_alice(row):
    """Returns distance between Alice and a row of the attributes table"""
    return distance(alice, np.array(row))

In [22]:
distance_from_alice(ckd_attributes.row(3))

1.4216649188818471

Now we can call the function `distance_from_alice` on each row of `ckd_attributes`, and augment the table `ckd` with the distances. Step 1 is complete!

In [23]:
distances = make_array()
for row in ckd_attributes.iter_rows():
    distances = np.append(distances, distance_from_alice(row))
ckd_with_distances = ckd.with_columns(pl.Series('Distance from Alice', distances))

In [24]:
ckd_with_distances

Hemoglobin,Glucose,Class,Color,Distance from Alice
f64,f64,i64,str,f64
0.456884,0.133751,0,"""gold""",1.068822
1.153004,-0.947597,0,"""gold""",2.349909
0.770138,-0.762223,0,"""gold""",2.015189
0.596108,-0.190654,0,"""gold""",1.421665
-0.239236,-0.49961,0,"""gold""",1.617401
…,…,…,…,…
-1.42264,1.693982,1,"""darkblue""",1.541661
-1.492252,1.27689,1,"""darkblue""",1.502699
-1.7707,-0.391475,1,"""darkblue""",2.315141


For Step 2, let's sort the table in increasing order of distance:

In [25]:
sorted_by_distance = ckd_with_distances.sort('Distance from Alice')
sorted_by_distance

Hemoglobin,Glucose,Class,Color,Distance from Alice
f64,f64,i64,str,f64
0.83975,1.215099,1,"""darkblue""",0.847601
-0.970162,1.27689,1,"""darkblue""",0.986156
-0.0304,0.087407,0,"""gold""",1.013049
0.14363,0.087407,0,"""gold""",1.022728
-0.413266,2.049282,1,"""darkblue""",1.035338
…,…,…,…,…
-1.004968,3.841231,1,"""darkblue""",2.919641
-2.29279,-0.85491,1,"""darkblue""",3.013065
-0.37846,4.520935,1,"""darkblue""",3.441806


Step 3: The top 5 rows correspond to Alice's 5 nearest neighbors; you can replace 5 by any other positive integer.

In [26]:
alice_5_nearest_neighbors = sorted_by_distance.head(5)
alice_5_nearest_neighbors

Hemoglobin,Glucose,Class,Color,Distance from Alice
f64,f64,i64,str,f64
0.83975,1.215099,1,"""darkblue""",0.847601
-0.970162,1.27689,1,"""darkblue""",0.986156
-0.0304,0.087407,0,"""gold""",1.013049
0.14363,0.087407,0,"""gold""",1.022728
-0.413266,2.049282,1,"""darkblue""",1.035338


Three of Alice's five nearest neighbors are blue points and two are gold. So a 5-nearest neighbor classifier would classify Alice as blue: it would predict that Alice has chronic kidney disease.

The graph below zooms in on Alice and her five nearest neighbors. The two gold ones just inside the circle directly below the red point. The classifier says Alice is more like the three blue ones around her.

In [27]:
radius = sorted_by_distance.get_column('Distance from Alice').item(4)+0.014
theta = np.arange(0, 2*np.pi+1, 2*np.pi/200)
(
    ckd.hvplot.scatter(x='Hemoglobin', y='Glucose', color='Color', cmap=color_map, size=40)
    * hv.Scatter(([alice.item(0)], [alice.item(1)])).opts(color='red', size=8)
    * hv.Curve((radius*np.cos(theta)+alice.item(0), radius*np.sin(theta)+alice.item(1))).opts(color='green', line_width=1.5)
).opts(xlim=(-2, 2.5), ylim=(-2, 2.5), frame_width=450, frame_height=450)

The previous scatterplot is zoomed in to the area surrounding the red data point at rough (0, 1). A green circle is drawn around this data point with the data point in the center. The green circle contains 3 dark blue data points and 2 gold data points.

We are well on our way to implementing our k-nearest neighbor classifier. In the next two sections we will put it together and assess its accuracy.